In [41]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [3]:
#文件已解压
#  import zipfile
# import os

# # 定义解压函数
# def unzip_file(zip_src, dst_dir):
#     if zipfile.is_zipfile(zip_src):
#         fz = zipfile.ZipFile(zip_src, 'r')
#         for file in fz.namelist():
#             fz.extract(file, dst_dir)
#         print(f"解压完成，文件已保存到 {dst_dir}")
#     else:
#         print('这不是一个有效的 zip 文件！')

# # 调用解压函数
# zip_file_path = '/root/autodl-fs/shouge_detection.zip'  # 替换为你的压缩文件路径
# destination_dir = '/root/autodl-fs'     # 替换为你想要解压到的目标路径

# # 确保目标目录存在
# os.makedirs(destination_dir, exist_ok=True)

# unzip_file(zip_file_path, destination_dir)

In [4]:
import os
import re

def clean_filename(filename):
    """清理文件名：移除空格和括号"""
    # 示例转换："类别_jpg (1).jpg" -> "类别_jpg_1.jpg"
    new_name = re.sub(r'[ ()]', '_', filename)  # 替换所有空格和括号为下划线
    new_name = re.sub(r'_+', '_', new_name)     # 合并连续的下划线
    new_name = new_name.replace('_.', '.')      # 处理扩展名前的多余下划线
    return new_name

def batch_rename_images(folder_path):
    """批量重命名文件夹中的图片"""
    renamed_files = {}
    
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            # 生成新文件名
            new_name = clean_filename(filename)
            
            # 重命名文件
            old_path = os.path.join(folder_path, filename)
            new_path = os.path.join(folder_path, new_name)
            os.rename(old_path, new_path)
            
            renamed_files[filename] = new_name
            print(f"Renamed: {filename} -> {new_name}")
    
    return renamed_files


if __name__ == "__main__":
    # 配置路径（修改为你的实际路径）
    image_folder = "/root/autodl-fs/shouge_detection/train"  # 图片文件夹
    # image_folder = "/root/autodl-fs/shouge_detection/val"  # 图片文件夹
    # 执行重命名
    renamed = batch_rename_images(image_folder)

   

Renamed: weishouge_jpg (192).jpg -> weishouge_jpg_192.jpg
Renamed: weishouge_jpg (15).jpg -> weishouge_jpg_15.jpg
Renamed: shouge_jpg_8.jpg -> shouge_jpg_8.jpg
Renamed: weishouge_jpg_35.jpg -> weishouge_jpg_35.jpg
Renamed: shouge_jpg_127.jpg -> shouge_jpg_127.jpg
Renamed: shouge_jpg_79.jpg -> shouge_jpg_79.jpg
Renamed: weishouge_jpg_50.jpg -> weishouge_jpg_50.jpg
Renamed: shouge_jpg_75.jpg -> shouge_jpg_75.jpg
Renamed: weishouge_jpg (129).jpg -> weishouge_jpg_129.jpg
Renamed: weishouge_jpg_250.jpg -> weishouge_jpg_250.jpg
Renamed: shouge_jpg (42).jpg -> shouge_jpg_42.jpg
Renamed: weishouge_jpg_196.jpg -> weishouge_jpg_196.jpg
Renamed: shouge_jpg_28.jpg -> shouge_jpg_28.jpg
Renamed: shouge_jpg (31).jpg -> shouge_jpg_31.jpg
Renamed: weishouge_jpg_97.jpg -> weishouge_jpg_97.jpg
Renamed: weishouge_jpg (148).jpg -> weishouge_jpg_148.jpg
Renamed: weishouge_jpg (165).jpg -> weishouge_jpg_165.jpg
Renamed: weishouge_jpg (158).jpg -> weishouge_jpg_158.jpg
Renamed: weishouge_jpg_220.jpg -> weisho

In [40]:
import os

def add_path_to_label_file(label_file_path, image_folder_path, output_file_path):
    """
    为标签文件中的图片名添加路径，并保存到新的文件中。
    
    :param label_file_path: 原始标签文件路径
    :param image_folder_path: 图片文件夹路径
    :param output_file_path: 输出文件路径
    """
    with open(label_file_path, 'r') as f:
        lines = f.readlines()
    
    with open(output_file_path, 'w') as f:
        for line in lines:
            parts = line.strip().split(' ')
            if len(parts) != 2:
                print(f"Skipping invalid line: {line.strip()}")
                continue
            image_name, label = parts
            image_path = os.path.join(image_folder_path, image_name)
            if os.path.exists(image_path):
                f.write(f"{image_path}\t{label}\n")
            else:
                print(f"Skipping missing image: {image_path}")

# # 定义原始标签文件路径、图片文件夹路径和输出文件路径
# label_file_path = '/root/autodl-fs/shouge_detection/val_labels.txt'  # 原始标签文件路径
# image_folder_path = '/root/autodl-fs/shouge_detection/images'  # 图片文件夹路径
# output_file_path = '/root/autodl-fs/shouge_detection/val_labels2.txt'  # 输出文件路径

# 定义原始标签文件路径、图片文件夹路径和输出文件路径
label_file_path = '/root/autodl-fs/shouge_detection/train_labels.txt'  # 原始标签文件路径
image_folder_path = '/root/autodl-fs/shouge_detection/images'  # 图片文件夹路径
output_file_path = '/root/autodl-fs/shouge_detection/train_labels2.txt'  # 输出文件路径

# 调用函数生成新的标签文件
add_path_to_label_file(label_file_path, image_folder_path, output_file_path)

In [ ]:
# 参数配置
class Config:
    input_size = [3, 224, 224]  # 输入图片的shape
    class_dim = 2  # 分类数（倒伏/未倒伏）
    data_path = "./shouge_detection/"  # 数据集路径
    train_list_path = "./shouge_detection/train_labels2.txt"  # 训练集列表
    eval_list_path = "./shouge_detection/val_labels2.txt"  # 验证集列表
    num_epochs = 50  # 训练轮数
    batch_size = 16  # 批次大小
    learning_rate = 0.0001  # 学习率
    checkpoint_dir = "./checkpoints/"  # 模型保存路径
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
# 检查可用的 GPU 设备
if torch.cuda.is_available():
    print("Available GPU devices:", torch.cuda.device_count())
    print("Current GPU device:", torch.cuda.current_device())
    print("Current GPU device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print("No GPU available. Using CPU for training.")

Available GPU devices: 1
Current GPU device: 0
Current GPU device name: NVIDIA GeForce RTX 4090


In [36]:
!ls

checkpoints	   prediction_results.txt  shouge_detection.zip
main_shouge.ipynb  shouge_detection	   test_pictures


In [ ]:
class WheatDataset(Dataset):
    def __init__(self, data_path, mode='train'):
        super().__init__()
        self.data_path = data_path
        self.img_paths = []
        self.labels = []
        
        list_path = Config.train_list_path if mode == 'train' else Config.eval_list_path
        
        with open(list_path, 'r') as f:
            for line in f.readlines():
                img_path, label = line.strip().split('\t')
                self.img_paths.append(img_path)
                self.labels.append(int(label))
        
        # 数据增强和归一化
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

    def __getitem__(self, index):
        img_path = self.img_paths[index]
        try:
            img = Image.open(img_path).convert('RGB')
            img = self.transform(img)
            label = torch.tensor(self.labels[index], dtype=torch.long)
            return img, label
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # 跳过错误图片，返回一个随机生成的图片和标签
            return torch.randn(3, 224, 224), torch.tensor(0, dtype=torch.long)

    def __len__(self):
        return len(self.img_paths)

In [31]:
# 定义ResNet50模型
class WheatResNet50(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.resnet = models.resnet50(pretrained=True)
        # 修改最后一层全连接层
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.resnet(x)

In [32]:
def train_model():
    # 创建数据集和数据加载器
    train_dataset = WheatDataset(Config.data_path, mode='train')
    eval_dataset = WheatDataset(Config.data_path, mode='eval')
    
    train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True)
    eval_loader = DataLoader(eval_dataset, batch_size=Config.batch_size, shuffle=False)
    
    # 初始化模型
    model = WheatResNet50(num_classes=Config.class_dim).to(Config.device)
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=Config.learning_rate)
    
    # 训练循环
    best_acc = 0.0
    for epoch in range(Config.num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for i, (inputs, labels) in enumerate(train_loader):
            # 检查是否为随机生成的图片
            if inputs.shape[0] != Config.batch_size:
                print(f"Skipping batch {i+1} due to incomplete batch size.")
                continue
            
            inputs = inputs.to(Config.device)
            labels = labels.to(Config.device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # 统计信息
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if (i+1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{Config.num_epochs}], Step [{i+1}/{len(train_loader)}], '
                      f'Loss: {loss.item():.4f}')
        
        # 每个epoch结束后在验证集上评估
        train_acc = 100 * correct / total
        eval_loss, eval_acc = evaluate_model(model, eval_loader)
        
        print(f'Epoch [{epoch+1}/{Config.num_epochs}], '
              f'Train Loss: {running_loss/len(train_loader):.4f}, '
              f'Train Acc: {train_acc:.2f}%, '
              f'Eval Loss: {eval_loss:.4f}, '
              f'Eval Acc: {eval_acc:.2f}%')
        
        # 保存最佳模型
        if eval_acc > best_acc:
            best_acc = eval_acc
            torch.save(model.state_dict(), os.path.join(Config.checkpoint_dir, 'best_model.pth'))
            print(f'Best model saved with accuracy: {best_acc:.2f}%')
    
    print('Training finished!')

In [33]:
# 评估函数
def evaluate_model(model, data_loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(Config.device)
            labels = labels.to(Config.device)
            
            outputs = model(inputs)
            loss = nn.CrossEntropyLoss()(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(data_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


In [34]:
# 预测函数（将结果写入txt并打印）
def predict_and_save_results(model, image_folder, output_file):
    model.eval()
    results = []
    
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225])
    ])
    
    class_names = ['未收割', '已收割']  # 根据实际类别顺序调整
    
    with torch.no_grad():
        for img_name in os.listdir(image_folder):
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            img_path = os.path.join(image_folder, img_name)
            img = Image.open(img_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(Config.device)
            
            output = model(img_tensor)
            prob = torch.softmax(output, dim=1)
            _, pred = torch.max(output, 1)
            
            result = {
                'filename': img_name,
                'class_id': pred.item(),
                'class_name': class_names[pred.item()],
                'probability': prob[0][pred.item()].item()
            }
            results.append(result)
            
            # 打印结果
            print(f"{img_name}: {class_names[pred.item()]} (置信度: {prob[0][pred.item()]:.2%})")
    
    # 写入txt文件
    with open(output_file, 'w') as f:
        for res in results:
            f.write(f"{res['filename']}\t{res['class_id']}\t{res['class_name']}\t{res['probability']:.4f}\n")
    
    print(f"预测结果已保存到: {output_file}")


In [17]:
import os
import shutil

def batch_rename_and_move_files(source_dir, target_dir, prefix):
    """
    批量修改文件名并移动到目标目录。
    
    :param source_dir: 包含文件的源目录路径
    :param target_dir: 目标目录路径
    :param prefix: 添加到文件名前的前缀
    """
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)  # 如果目标目录不存在，则创建它

    for filename in os.listdir(source_dir):
        source_file_path = os.path.join(source_dir, filename)
        if os.path.isfile(source_file_path):
            # 构造新的文件名
            new_filename = f"{prefix}_{filename}"
            target_file_path = os.path.join(target_dir, new_filename)
            # 移动并重命名文件
            shutil.move(source_file_path, target_file_path)
            print(f"Moved and renamed: {filename} -> {new_filename}")

# 定义训练集和验证集的目录路径
train_dir = '/root/autodl-fs/shouge_detection/train'
val_dir = '/root/autodl-fs/shouge_detection/val'
images_dir = '/root/autodl-fs/shouge_detection/images'  # 目标目录

# 定义前缀
train_prefix = 'train'
val_prefix = 'val'

# 调用函数批量修改文件名并移动文件
batch_rename_and_move_files(train_dir, images_dir, train_prefix)
batch_rename_and_move_files(val_dir, images_dir, val_prefix)

Moved and renamed: shouge_jpg_8.jpg -> train_shouge_jpg_8.jpg
Moved and renamed: weishouge_jpg_35.jpg -> train_weishouge_jpg_35.jpg
Moved and renamed: shouge_jpg_127.jpg -> train_shouge_jpg_127.jpg
Moved and renamed: shouge_jpg_79.jpg -> train_shouge_jpg_79.jpg
Moved and renamed: weishouge_jpg_50.jpg -> train_weishouge_jpg_50.jpg
Moved and renamed: shouge_jpg_75.jpg -> train_shouge_jpg_75.jpg
Moved and renamed: weishouge_jpg_250.jpg -> train_weishouge_jpg_250.jpg
Moved and renamed: weishouge_jpg_196.jpg -> train_weishouge_jpg_196.jpg
Moved and renamed: shouge_jpg_28.jpg -> train_shouge_jpg_28.jpg
Moved and renamed: weishouge_jpg_97.jpg -> train_weishouge_jpg_97.jpg
Moved and renamed: weishouge_jpg_220.jpg -> train_weishouge_jpg_220.jpg
Moved and renamed: weishouge_jpg_105.jpg -> train_weishouge_jpg_105.jpg
Moved and renamed: weishouge_jpg_32.jpg -> train_weishouge_jpg_32.jpg
Moved and renamed: weishouge_jpg_133.jpg -> train_weishouge_jpg_133.jpg
Moved and renamed: shouge_jpg_131.jpg ->

In [38]:
if __name__ == "__main__":
    # 创建检查点目录
    os.makedirs(Config.checkpoint_dir, exist_ok=True)
    
    # 训练模型
    train_model()
    
    # 加载最佳模型进行预测
    model = WheatResNet50(num_classes=Config.class_dim).to(Config.device)
    model.load_state_dict(torch.load(os.path.join(Config.checkpoint_dir, 'best_model.pth')))
    
    # 预测测试集并保存结果
    test_images_path = "/root/autodl-fs/test_pictures/test_pictures_1/"  # 测试图片目录
    output_txt = "./prediction_results.txt"  # 结果输出文件
    predict_and_save_results(model, test_images_path, output_txt)

/root/miniconda3/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/miniconda3/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Error loading image train_shouge_jpg_26.jpg: [Errno 2] No such file or directory: 'train_shouge_jpg_26.jpg'
Error loading image train_shouge_jpg_136.jpg: [Errno 2] No such file or directory: 'train_shouge_jpg_136.jpg'
Error loading image train_weishouge_jpg_237.jpg: [Errno 2] No such file or directory: 'train_weishouge_jpg_237.jpg'
Error loading image train_weishouge_jpg_187.jpg: [Errno 2] No such file or directory: 'train_weishouge_jpg_187.jpg'
Error loading image train_shouge_jpg_18.jpg: [Errno 2] No such file or directory: 'train_shouge_jpg_18.jpg'
Error loading image train_weishouge_jpg_219.jpg: [Errno 2] No such file or directory: 'train_weishouge_jpg_219.jpg'
Error loading image train_weishouge_jpg_17.jpg: [Errno 2] No such file or directory: 'train_weishouge_jpg_17.jpg'
Error loading image train_weishouge_jpg_235.jpg: [Errno 2] No such file or directory: 'train_weishouge_jpg_235.jpg'
Error loading image train_weishouge_jpg_25.jpg: [Errno 2] No such file or directory: 'train_weis